In [1]:
import sys
sys.path.append('../../')
import os
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from rover_simulator.navigation.rl_env import RoverGymEnv
from rl.callbacks import EpisodeInfoCallback, RolloutCheckpointCallback
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList

In [2]:
def make_env(map_file=None):
    def _init():
        return RoverGymEnv(map_file=map_file)
    return _init

In [3]:
timesteps = 100000
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "sac_rover")

In [4]:
env = DummyVecEnv([make_env()])
env = VecMonitor(env)   # Wrap with VecMonitor to record episode rewards/lengths for TensorBoard

model = SAC(
    "MlpPolicy",
    env,
    verbose=0,
    tensorboard_log="./rl_logs",
    batch_size=256, buffer_size=100000,
    learning_rate=3e-4
)

callbacks = []
checkpoint_cb = CheckpointCallback(save_freq=10000, save_path=model_dir, name_prefix="sac_rover")
# rollout_cb = RolloutCheckpointCallback(save_dir=model_dir, name_prefix="sac_rover")

callbacks.append(EpisodeInfoCallback())
callbacks.append(checkpoint_cb)
# callbacks.append(rollout_cb)
cb_list = CallbackList(callbacks)

In [5]:
model.learn(total_timesteps=timesteps, callback=cb_list, progress_bar=True)

Output()

In [6]:
model.save(model_path)